In [302]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer

# Import and select the data

In [303]:
df = pd.read_csv("data/combined_h1_2018_2025_excl_nan.csv")
df.head()

,Datetime,Location,NO2,O3,PM10,Location_Code,Hr,RainDur,T,WVv,StrGlo,Year,Date,Time,Month,Weekday,YearMonth,meanVehicles
0,2017-12-31 23:00:00+00:00,Zch_Rosengartenstrasse,17.20,44.33,18.81,1,67.36,0.00,9.02,1.85,0.02,2018,2017-12-31,23:00:00,12,6,2018-12-01,538.5
1,2017-12-31 23:00:00+00:00,Zch_Schimmelstrasse,9.33,49.89,16.57,2,64.18,0.00,9.60,0.70,0.02,2018,2017-12-31,23:00:00,12,6,2018-12-01,340.0
2,2017-12-31 23:00:00+00:00,Zch_Stampfenbachstrasse,12.80,46.59,27.35,3,66.37,0.00,9.06,4.62,0.02,2018,2017-12-31,23:00:00,12,6,2018-12-01,197.5
3,2018-01-01 00:00:00+00:00,Zch_Rosengartenstrasse,24.10,41.80,19.33,1,69.54,0.27,9.28,1.59,0.01,2018,2018-01-01,00:00:00,1,0,2018-01-01,1029.0
4,2018-01-01 00:00:00+00:00,Zch_Schimmelstrasse,8.67,50.66,11.73,2,67.29,0.00,9.53,0.99,0.01,2018,2018-01-01,00:00:00,1,0,2018-01-01,621.5


In [304]:
# Re-arrange columns
cols = ["Datetime", "YearMonth", "Date", "Year", "Month", "Weekday", "Time",
        "Location", "Location_Code",
        "NO2", "O3", "PM10",
        "Hr", "RainDur", "T", "WVv", "StrGlo",
        "meanVehicles"]

df = df[cols]
df.head()

,Datetime,YearMonth,Date,Year,Month,Weekday,Time,Location,Location_Code,NO2,O3,PM10,Hr,RainDur,T,WVv,StrGlo,meanVehicles
0,2017-12-31 23:00:00+00:00,2018-12-01,2017-12-31,2018,12,6,23:00:00,Zch_Rosengartenstrasse,1,17.20,44.33,18.81,67.36,0.00,9.02,1.85,0.02,538.5
1,2017-12-31 23:00:00+00:00,2018-12-01,2017-12-31,2018,12,6,23:00:00,Zch_Schimmelstrasse,2,9.33,49.89,16.57,64.18,0.00,9.60,0.70,0.02,340.0
2,2017-12-31 23:00:00+00:00,2018-12-01,2017-12-31,2018,12,6,23:00:00,Zch_Stampfenbachstrasse,3,12.80,46.59,27.35,66.37,0.00,9.06,4.62,0.02,197.5
3,2018-01-01 00:00:00+00:00,2018-01-01,2018-01-01,2018,1,0,00:00:00,Zch_Rosengartenstrasse,1,24.10,41.80,19.33,69.54,0.27,9.28,1.59,0.01,1029.0
4,2018-01-01 00:00:00+00:00,2018-01-01,2018-01-01,2018,1,0,00:00:00,Zch_Schimmelstrasse,2,8.67,50.66,11.73,67.29,0.00,9.53,0.99,0.01,621.5


In [305]:
# Add day
df["Day"] = df["Date"].str[-2:]

# Select columns
cols_selected = ["Year", "Month", "Day", "Weekday", "Time",
                 "Location",
                 "NO2", "O3", "PM10",
                 "Hr", "RainDur", "T", "WVv", "StrGlo",
                 "meanVehicles"]

df_selected = df[cols_selected]
df_selected.head()

,Year,Month,Day,Weekday,Time,Location,NO2,O3,PM10,Hr,RainDur,T,WVv,StrGlo,meanVehicles
0,2018,12,31,6,23:00:00,Zch_Rosengartenstrasse,17.20,44.33,18.81,67.36,0.00,9.02,1.85,0.02,538.5
1,2018,12,31,6,23:00:00,Zch_Schimmelstrasse,9.33,49.89,16.57,64.18,0.00,9.60,0.70,0.02,340.0
2,2018,12,31,6,23:00:00,Zch_Stampfenbachstrasse,12.80,46.59,27.35,66.37,0.00,9.06,4.62,0.02,197.5
3,2018,1,01,0,00:00:00,Zch_Rosengartenstrasse,24.10,41.80,19.33,69.54,0.27,9.28,1.59,0.01,1029.0
4,2018,1,01,0,00:00:00,Zch_Schimmelstrasse,8.67,50.66,11.73,67.29,0.00,9.53,0.99,0.01,621.5


# Prepare data for ML

## Preprocessing - OneHotEncoder, StandardScaler, Trigonometric encoding and SplineTransformer

In [306]:
# Prepare data transformer to be used in pipeline - do not use on the whole dataset!
# Encodings must be computed only on train data and then only applied on test data
# Trigonometric encoding could be in theory be done on the whole dataset, but for simplicity it is combined to have only one transformer

# Prepare Trigonometric encoding for time (safe before split)
def sin_transformer(period):
    return FunctionTransformer(lambda x: np.sin(x / period * 2 * np.pi))

def cos_transformer(period):
    return FunctionTransformer(lambda x: np.cos(x / period * 2 * np.pi))

# Prepare Spline Transformer
def periodic_spline_transformer(period, n_splines):
    degree = 3
    n_knots = n_splines + 1  # since include_bias is True
    return SplineTransformer(
        degree=degree,
        n_knots=n_knots,
        knots=np.linspace(0, period, n_knots).reshape(n_knots, 1),
        extrapolation="periodic",
        include_bias=True,
    )

# Prepare columns to be used
numeric_variables = ["NO2", "O3", "PM10", "Hr", "RainDur", "T", "WVv", "StrGlo", "meanVehicles"]

# Make sure the time features are numbers, not strings
df_converted = df_selected.copy()
df_converted["Time"] = pd.to_datetime(df_converted["Time"], format="%H:%M:%S").dt.hour.astype("int64")
df_converted["Month"] = df_selected["Month"].astype(int)
df_converted["Day"] = df_selected["Day"].astype(int)
df_converted["Weekday"] = df_selected["Weekday"].astype(int)

preprocess = ColumnTransformer(
    transformers=[
        ("Month_sin", sin_transformer(12), ["Month"]),
        ("Month_cos", cos_transformer(12), ["Month"]),
        ("Day_sin", sin_transformer(7), ["Day"]),
        ("Day_cos", cos_transformer(7), ["Day"]),
        ("Weekday_sin", sin_transformer(7), ["Weekday"]),
        ("Weekday_cos", cos_transformer(7), ["Weekday"]),
        ("Time_sin", sin_transformer(24), ["Time"]),
        ("Time_cos", cos_transformer(24), ["Time"]),
        ("Spline_month", periodic_spline_transformer(12, n_splines=5), ["Month"]),
        ("Spline_day", periodic_spline_transformer(7, n_splines=6), ["Day"]),
        ("Spline_weekday", periodic_spline_transformer(7, n_splines=6), ["Weekday"]),
        ("Spline_time", periodic_spline_transformer(24, n_splines=24), ["Time"]),
        ("encoded", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Location"]),
        ("scaled", StandardScaler(), numeric_variables),
    ],
    remainder="passthrough"
).set_output(transform="pandas")

# TODO: Delete later, here only as example use and the whole dataset for testing
df_transformed = preprocess.fit_transform(df_converted)
df_transformed = df_transformed.rename(
    columns=lambda c: c.replace("__Month", "")
        .replace("__Day", "")
        .replace("__Weekday", "")
        .replace("__Time", "")
        .replace("remainder__", "")
)

df_transformed.head()

,Month_sin,Month_cos,Day_sin,Day_cos,Weekday_sin,Weekday_cos,Time_sin,Time_cos,Spline_month_sp_0,Spline_month_sp_1,...,scaled__NO2,scaled__O3,scaled__PM10,scaled__Hr,scaled__RainDur,scaled__T,scaled__WVv,scaled__StrGlo,scaled__meanVehicles,Year
0,-2.449294e-16,1.000000,0.433884,-0.900969,-0.781831,0.62349,-0.258819,0.965926,0.166667,0.666667,...,-0.682185,-0.052938,0.431353,-0.128157,-0.345975,-0.436262,1.139169,-0.616970,-0.145770,2018
1,-2.449294e-16,1.000000,0.433884,-0.900969,-0.781831,0.62349,-0.258819,0.965926,0.166667,0.666667,...,-1.144052,0.130613,0.194810,-0.307204,-0.345975,-0.363149,-0.109481,-0.616970,-0.563023,2018
2,-2.449294e-16,1.000000,0.433884,-0.900969,-0.781831,0.62349,-0.258819,0.965926,0.166667,0.666667,...,-0.940408,0.021671,1.333171,-0.183898,-0.345975,-0.431219,4.146787,-0.616970,-0.862562,2018
3,5.000000e-01,0.866025,0.781831,0.623490,0.000000,1.00000,0.000000,1.000000,0.033083,0.529225,...,-0.277244,-0.136461,0.486264,-0.005414,-0.326709,-0.403487,0.856866,-0.617014,0.885274,2018
4,5.000000e-01,0.866025,0.781831,0.623490,0.000000,1.00000,0.000000,1.000000,0.033083,0.529225,...,-1.182786,0.156033,-0.316290,-0.132098,-0.345975,-0.371973,0.205396,-0.617014,0.028698,2018


## Split for Train and Test - fit and transform the Preprocessing transformer